# 04 수요패턴 분석 (개요)

매장 **type**별 수요 변동성을 간략히 살펴봅니다.

> **상세 분석**(System/SKU-Level 기초통계, SBC·ML 클러스터별 변동성, **RIDR 지수**, 수요 밴드 시각화)은  
> **`11_제품_수요패턴_RIDR_분석.ipynb`** 에서 논문 3장 스타일로 수행합니다.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))
from utils.paths import DATA_PROCESSED
from utils.stats_summary import system_level_weekly_stats, sku_level_weekly_stats
from utils.sbc import compute_sbc_table

dfw = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
types = sorted(dfw['type'].unique())
print('type 개수:', len(types), types)



In [ ]:
# type별 System-Level / SKU-Level 기초통계 (논문 Table 3.2 스타일)
system_rows, sku_rows, sku_detail_all = [], [], []
for t in types:
    sys_df = system_level_weekly_stats(dfw, t)
    sku_sum, sku_detail = sku_level_weekly_stats(dfw, t)
    system_rows.append(sys_df)
    sku_rows.append(sku_sum)
    sku_detail_all.append(sku_detail)

system_stats = pd.concat(system_rows, ignore_index=True)
sku_stats = pd.concat(sku_rows, ignore_index=True)
sku_detail = pd.concat(sku_detail_all, ignore_index=True)

print('=== System-Level (type별 주간 총판매) ===')
print(system_stats.round(3))
print('\n=== SKU-Level 요약 (type별 family 시계열 평균 통계) ===')
print(sku_stats.round(3))



In [ ]:
# type별 ADI/CV2 분포 (주간 sales 기준, type×family)
sbc_preview = compute_sbc_table(dfw, ['type','family'], qty_col='sales')
print(sbc_preview.groupby(['type','SBC_CLUSTER_LABEL']).size().unstack(fill_value=0))
sbc_preview.head(10)



In [ ]:
# type별 변동성(CV) 비교 → 이후 SBC vs ML 선택 근거
vol = sku_detail.groupby('type')['cv'].mean().sort_values()
print('type별 평균 CV (낮을수록 안정):')
print(vol)
vol.plot(kind='bar', title='Type별 평균 CV (SKU-Level)')
plt.ylabel('CV')
plt.tight_layout()
plt.show()



In [ ]:
# 저장
out_dir = DATA_PROCESSED
system_stats.to_csv(out_dir / 'type_system_level_stats.csv', index=False)
sku_stats.to_csv(out_dir / 'type_sku_level_stats.csv', index=False)
sku_detail.to_csv(out_dir / 'type_family_series_stats.csv', index=False)
print('저장 완료')



## 분석 요약 (개요)

### System-Level (type별 주간 총판매)
- **type D·A** 규모 최대 (주간 평균 ~145만), **type E** 최소 (~24.7만)
- CV는 **E(0.43) > B(0.34) > D·A(0.30)** → 소규모 type일수록 상대 변동성 큼
- 왜도·첨도 모두 음수에 가까워 **왼쪽 꼬리가 짧은 분포** (극단적 저수요 주가 적음)

### SKU-Level (family 시계열 평균)
- type당 **33개 family** 시계열, 평균 CV는 **E(0.87) > C(0.72) > A(0.65)**
- type D가 평균 CV 최저(0.62) → **가장 안정적인 매장 유형**

### SBC 패턴 미리보기 (type×family)
- 모든 type에서 **Smooth(19개)** 가 가장 많고, **Intermittent(8~10개)** 가 그다음
- Erratic·Lumpy는 type당 1~3개로 소수

> 상세 분석(System/SKU 클러스터 통계, **RIDR**, 수요 밴드)은 **`11_제품_수요패턴_RIDR_분석.ipynb`** 참조